TP 1 - Spark RDD

Noms : Nicolas Rihana et Benjamin Labos - groupe 4


In [1]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8


File ‘around_the_world_in_80_days.txt’ already there; not retrieving.


# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [2]:
# start a spark session and create spark context for making rdd
import importlib.util
import os
import sys
from pathlib import Path

if importlib.util.find_spec("pyspark") is None:
    spark_home = Path(os.environ.get("SPARK_HOME", "/usr/local/spark"))
    spark_python = spark_home / "python"
    py4j_files = list((spark_python / "lib").glob("py4j-*-src.zip"))
    if not spark_python.exists() or not py4j_files:
        raise RuntimeError("PySpark introuvable dans l'image Docker")
    paths = [str(spark_python), str(py4j_files[0])]
    sys.path[:0] = paths
    existing_path = os.environ.get("PYTHONPATH")
    os.environ["PYTHONPATH"] = os.pathsep.join(
        paths + ([existing_path] if existing_path else [])
    )
    os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[2]")
         .appName("TP2_RDD_word_count")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark", spark.version)


Spark 4.2.0


In [3]:
# Defind the rdd
from pathlib import Path
import re


def book_body(filename, first_line, last_line):
    raw_path = Path(filename)
    lines = raw_path.read_text(encoding="utf-8-sig").splitlines()
    starts = [i for i, line in enumerate(lines)
              if line.strip() == first_line]
    ends = [i for i, line in enumerate(lines)
            if line.strip() == last_line]
    if len(starts) != 1 or len(ends) != 1 or starts[0] >= ends[0]:
        raise ValueError(f"Book markers not found in {filename}")
    body_path = raw_path.with_name(raw_path.stem + "_body.txt")
    body_path.write_text(
        "\n".join(lines[starts[0]:ends[0]]) + "\n", encoding="utf-8"
    )
    return body_path


english_path = book_body(
    "around_the_world_in_80_days.txt",
    "CHAPTER I.",
    "*** END OF THE PROJECT GUTENBERG EBOOK AROUND THE WORLD IN EIGHTY DAYS ***",
)
rdd = sc.textFile(str(english_path), minPartitions=2)


In [4]:
# view the first x lines of the rdd
rdd.take(20)


['CHAPTER I.',
 'IN WHICH PHILEAS FOGG AND PASSEPARTOUT ACCEPT EACH OTHER, THE ONE AS',
 'MASTER, THE OTHER AS MAN',
 '',
 '',
 'Mr. Phileas Fogg lived, in 1872, at No. 7, Saville Row, Burlington',
 'Gardens, the house in which Sheridan died in 1814. He was one of the',
 'most noticeable members of the Reform Club, though he seemed always to',
 'avoid attracting attention; an enigmatical personage, about whom little',
 'was known, except that he was a polished man of the world. People said',
 'that he resembled Byron—at least that his head was Byronic; but he was',
 'a bearded, tranquil Byron, who might live on a thousand years without',
 'growing old.',
 '',
 'Certainly an Englishman, it was more doubtful whether Phileas Fogg was',
 'a Londoner. He was never seen on ’Change, nor at the Bank, nor in the',
 'counting-rooms of the “City”; no ships ever came into London docks of',
 'which he was the owner; he had no public employment; he had never been',
 'entered at any of the Inns of Co

In [5]:
# example lambda function
words = rdd.flatMap(lambda line: line.split())


In [6]:
# Note and explain the output of the below command
words


PythonRDD[3] at RDD at PythonRDD.scala:59

La variable words ne montre pas les mots. Spark les calcule quand on utilise une action comme take(20).


La fonction lambda coupe chaque ligne en mots. flatMap met tous ces mots dans le même RDD.


In [7]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
collected_words = words.collect()
print("Nombre de tokens avant nettoyage :", len(collected_words))
collected_words[:20]


Nombre de tokens avant nettoyage : 62760


['CHAPTER',
 'I.',
 'IN',
 'WHICH',
 'PHILEAS',
 'FOGG',
 'AND',
 'PASSEPARTOUT',
 'ACCEPT',
 'EACH',
 'OTHER,',
 'THE',
 'ONE',
 'AS',
 'MASTER,',
 'THE',
 'OTHER',
 'AS',
 'MAN',
 'Mr.']

collect() lance le calcul et récupère tous les mots dans une liste. La case précédente montrait seulement le RDD.


In [8]:
# nicer print
for word in words.take(20):
    print(word)


CHAPTER
I.
IN
WHICH
PHILEAS
FOGG
AND
PASSEPARTOUT
ACCEPT
EACH
OTHER,
THE
ONE
AS
MASTER,
THE
OTHER
AS
MAN
Mr.


In [9]:
# Print first x words
words.take(20)


['CHAPTER',
 'I.',
 'IN',
 'WHICH',
 'PHILEAS',
 'FOGG',
 'AND',
 'PASSEPARTOUT',
 'ACCEPT',
 'EACH',
 'OTHER,',
 'THE',
 'ONE',
 'AS',
 'MASTER,',
 'THE',
 'OTHER',
 'AS',
 'MAN',
 'Mr.']

In [10]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.
?rdd.flatMap


In [11]:
# Initialize a word counter by creating a tuple with word and cound of 1
word_pairs = words.map(lambda word: (word, 1))
word_pairs.take(20)


[('CHAPTER', 1),
 ('I.', 1),
 ('IN', 1),
 ('WHICH', 1),
 ('PHILEAS', 1),
 ('FOGG', 1),
 ('AND', 1),
 ('PASSEPARTOUT', 1),
 ('ACCEPT', 1),
 ('EACH', 1),
 ('OTHER,', 1),
 ('THE', 1),
 ('ONE', 1),
 ('AS', 1),
 ('MASTER,', 1),
 ('THE', 1),
 ('OTHER', 1),
 ('AS', 1),
 ('MAN', 1),
 ('Mr.', 1)]

In [12]:
# a. count the occurence of each word
from operator import add

raw_counts = word_pairs.reduceByKey(add)
raw_counts.take(20)


[('I.', 1),
 ('IN', 39),
 ('FOGG', 17),
 ('EACH', 2),
 ('MASTER,', 2),
 ('OTHER', 1),
 ('MAN', 1),
 ('Mr.', 371),
 ('Phileas', 214),
 ('lived,', 1),
 ('1872,', 2),
 ('at', 563),
 ('Row,', 7),
 ('Burlington', 1),
 ('Gardens,', 1),
 ('Sheridan', 1),
 ('died', 2),
 ('of', 1764),
 ('most', 39),
 ('though', 30)]

In [13]:
# b. a common first step in text analysis, change all capital letters to lower case
lower_words = rdd.flatMap(lambda line: line.lower().split())
lower_counts = lower_words.map(lambda word: (word, 1)).reduceByKey(add)
lower_counts.take(20)


[('chapter', 37),
 ('i.', 1),
 ('phileas', 232),
 ('fogg', 348),
 ('and', 1761),
 ('passepartout', 229),
 ('accept', 2),
 ('as', 426),
 ('master,', 22),
 ('other', 44),
 ('man', 47),
 ('lived,', 1),
 ('1872,', 2),
 ('at', 626),
 ('saville', 17),
 ('row,', 7),
 ('died', 2),
 ('of', 1786),
 ('most', 39),
 ('though', 30)]

In [14]:
# c. eliminate the stop words.
STOP_EN = frozenset("""
a an and are as at be been but by can could did do does for from had has
have he her hers him his how i if in into is it its me my no nor not of
on or our ours out over she so some than that the their them there these
they this those through to too under until up us was we were what when
where which while who whom why will with would you your yours am being
all any both each few more most other same such very own only then now
about after again against before between down during further here once
above below off should must just might may shall itself himself herself
themselves yourself ourselves i'm i've i'll i'd you're you've you'll
we're we've we'll we'd he's she'll she's it's they're they've they'll
that's there's what's who's where's don't doesn't didn't isn't aren't
wasn't weren't hasn't haven't hadn't won't wouldn't shouldn't couldn't
shan't mustn't let's
""".split())

filtered_words = lower_words.filter(lambda word: word not in STOP_EN)
filtered_counts = filtered_words.map(lambda word: (word, 1))\
                               .reduceByKey(add)
filtered_counts.take(20)


[('chapter', 37),
 ('i.', 1),
 ('phileas', 232),
 ('fogg', 348),
 ('passepartout', 229),
 ('accept', 2),
 ('master,', 22),
 ('man', 47),
 ('lived,', 1),
 ('1872,', 2),
 ('saville', 17),
 ('row,', 7),
 ('died', 2),
 ('though', 30),
 ('always', 22),
 ('avoid', 6),
 ('enigmatical', 1),
 ('little', 46),
 ('polished', 3),
 ('resembled', 3)]

In [15]:
# d. sort in alphabetical order
alphabetical_counts = filtered_counts.sortByKey()
alphabetical_counts.take(20)


[('&c.,', 1),
 ('(japan),', 1),
 ('(saturday,', 1),
 ('(sort', 1),
 ('(sunday)', 1),
 ('-------', 1),
 ('.', 3),
 ('.....', 1),
 ('........', 1),
 ('.........', 1),
 ('.............', 2),
 ('.................', 1),
 ('...................', 1),
 ('....................', 1),
 ('............................................', 1),
 ('1,000.', 1),
 ('1,170,', 1),
 ('10th,', 1),
 ('11', 1),
 ('11.40', 1)]

In [16]:
# e. sort descending by word frequency
frequency_counts = filtered_counts.sortBy(
    lambda pair: (-pair[1], pair[0])
)
frequency_counts.take(20)


[('mr.', 372),
 ('fogg', 348),
 ('phileas', 232),
 ('passepartout', 229),
 ('said', 156),
 ('fogg,', 131),
 ('one', 131),
 ('fix', 125),
 ('passepartout,', 119),
 ('“i', 115),
 ('upon', 111),
 ('two', 97),
 ('hundred', 91),
 ('replied', 89),
 ('and,', 83),
 ('thousand', 83),
 ('time', 82),
 ('made', 81),
 ('like', 80),
 ('without', 77)]

In [17]:
# f. remove punctuations and blank spaces
def tokenize(line, split_apostrophes=False):
    line = line.lower().replace("’", "'")
    if split_apostrophes:
        return re.findall(r"[^\W\d_]+", line)
    return re.findall(r"[^\W\d_]+(?:'[^\W\d_]+)*", line)


def count_words(lines, stopwords, split_apostrophes=False):
    return (lines
            .flatMap(lambda line: tokenize(line, split_apostrophes))
            .filter(lambda word: word not in stopwords)
            .map(lambda word: (word, 1))
            .reduceByKey(add))


clean_counts_en = count_words(rdd, STOP_EN)
print("Alphabetical:", clean_counts_en.sortByKey().take(20))
print("Most frequent:", clean_counts_en.sortBy(
    lambda pair: (-pair[1], pair[0])
).take(20))


Alphabetical: [('abandon', 5), ('abandoned', 5), ('abandoning', 1), ('abashed', 1), ('abduction', 2), ('aberration', 1), ('able', 18), ('ablutions', 1), ('aboard', 4), ('abode', 1), ('abolishing', 1), ('abominable', 2), ('abominably', 1), ('abraham', 1), ('abridged', 1), ('abroad', 1), ('abrupt', 2), ('abruptly', 1), ('absence', 1), ('absented', 1)]


Most frequent: [('fogg', 584), ('passepartout', 392), ('mr', 390), ('phileas', 238), ('fix', 234), ('said', 193), ('one', 169), ('time', 126), ('aouda', 125), ('train', 119), ('upon', 119), ('two', 107), ('sir', 103), ('master', 102), ('hundred', 97), ('twenty', 96), ('well', 96), ('replied', 93), ('hours', 90), ('day', 89)]


# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [18]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([
    ("Brooke", 20), ("Denny", 31), ("Jules", 30),
    ("TD", 35), ("Brooke", 25)
])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
    # Keep the name and age, and add a count of 1.
    .map(lambda x: (x[0], (x[1], 1)))
    # Add the ages and count the people with the same name.
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
    # Divide the total age by the number of people.
    .map(lambda x: (x[0], x[1][0] / x[1][1])))

age_results = dict(agesRDD.collect())
print(sorted(age_results.items()))
assert age_results["Brooke"] == 22.5


[('Brooke', 22.5), ('Denny', 31.0), ('Jules', 30.0), ('TD', 35.0)]


## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [19]:
# Code here
import statistics
import time


def late_filter(lines, stopwords):
    return (lines
            .flatMap(tokenize)
            .map(lambda word: (word, 1))
            .reduceByKey(add)
            .filter(lambda pair: pair[0] not in stopwords))


def timed_result(build):
    start = time.perf_counter()
    result = dict(build().collect())
    return time.perf_counter() - start, result


rdd.cache()
rdd.count()
pipelines = {
    "A - filtre avant reduceByKey": lambda: count_words(rdd, STOP_EN),
    "B - filtre après reduceByKey": lambda: late_filter(rdd, STOP_EN),
}
reference = dict(pipelines["A - filtre avant reduceByKey"]().collect())
timings = {name: [] for name in pipelines}
names = list(pipelines)
for turn in range(3):
    for name in names[turn % 2:] + names[:turn % 2]:
        elapsed, result = timed_result(pipelines[name])
        assert result == reference
        timings[name].append(elapsed)

medians = {name: statistics.median(values)
           for name, values in timings.items()}
for name, values in timings.items():
    print(name, "mesures (s):", [round(v, 3) for v in values],
          "médiane:", round(medians[name], 3))
best = min(medians, key=medians.get)
gap = (max(medians.values()) - min(medians.values())) / min(medians.values())
print("Médiane la plus basse sur ce PC:", best)
if gap < 0.10:
    print("Écart inférieur à 10 % : classement non concluant sur ce livre.")
print("Pour de gros textes, filtrer tôt réduit les données à agréger.")
rdd.unpersist()


A - filtre avant reduceByKey mesures (s): [0.288, 0.423, 0.337] médiane: 0.337
B - filtre après reduceByKey mesures (s): [0.307, 0.312, 1.263] médiane: 0.312
Médiane la plus basse sur ce PC: B - filtre après reduceByKey
Écart inférieur à 10 % : classement non concluant sur ce livre.
Pour de gros textes, filtrer tôt réduit les données à agréger.


around_the_world_in_80_days_body.txt MapPartitionsRDD[1] at textFile at DirectMethodHandleAccessor.java:103

## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two
Les deux livres sont nettoyés séparément. On compare ensuite le nombre de mots dans chaque langue.


In [20]:
# Code here
from urllib.request import urlretrieve

french_raw_path = Path("around_world_fr_46541.txt")
if not french_raw_path.exists():
    urlretrieve(
        "https://www.gutenberg.org/cache/epub/46541/pg46541.txt",
        french_raw_path,
    )
french_path = book_body(
    french_raw_path,
    "DANS LEQUEL PHILEAS FOGG ET PASSEPARTOUT S'ACCEPTENT RÉCIPROQUEMENT,",
    "FIN",
)
rdd_fr = sc.textFile(str(french_path), minPartitions=2)

STOP_FR = frozenset("""
à au aux avec ce ces cet cette dans de des du elle elles en et eux il ils
je la le les leur leurs lui ma mais me mes moi mon ne nos notre nous on
ont ou où par pas pour qu que quel quelle quelles quels qui sa sans se
ses son sont sous sur ta te tes toi ton tu un une vos votre vous y a ai
as avons avez avait avaient avais avoir eu eue eus eues est es suis
sommes êtes être été était étaient étais étiez étions sera seront serait
seraient plus moins très tout tous toute toutes dont comme si ni non bien
encore après avant chez ceux celles celle chaque jusque lorsque parce
peut peu fait faire fut fût soit c d j l m n s t
""".split())

counts_fr = count_words(rdd_fr, STOP_FR, split_apostrophes=True)
assert not counts_fr.filter(lambda pair: "'" in pair[0]).take(1)

total_en = int(clean_counts_en.values().sum())
total_fr = int(counts_fr.values().sum())
distinct_en = clean_counts_en.count()
distinct_fr = counts_fr.count()
print("Langue | mots conservés | vocabulaire distinct")
print("EN", total_en, distinct_en)
print("FR", total_fr, distinct_fr)

for name, counts in [("EN", clean_counts_en), ("FR", counts_fr)]:
    print(name, "top 20:", counts.sortBy(
        lambda pair: (-pair[1], pair[0])
    ).take(20))

comparison = (clean_counts_en.fullOuterJoin(counts_fr)
              .mapValues(lambda pair: (pair[0] or 0, pair[1] or 0)))
print("Mêmes graphies dans les deux textes:")
print(comparison.filter(
    lambda pair: pair[1][0] > 0 and pair[1][1] > 0
).sortBy(lambda pair: (-(pair[1][0] + pair[1][1]), pair[0]))
 .take(20))

names = {"fogg", "passepartout", "fix", "aouda"}
print("Nom | EN | FR | EN / 10 000 | FR / 10 000")
for word, (en, fr) in comparison.filter(
    lambda pair: pair[0] in names
).sortByKey().collect():
    print(word, en, fr,
          round(10000 * en / total_en, 2),
          round(10000 * fr / total_fr, 2))

print("Interprétation : les fréquences relatives tiennent compte des tailles")
print("différentes ; cette jointure ne traduit pas les mots.")


Langue | mots conservés | vocabulaire distinct
EN 31901 6693
FR 36894 8659


EN top 20: [('fogg', 584), ('passepartout', 392), ('mr', 390), ('phileas', 238), ('fix', 234), ('said', 193), ('one', 169), ('time', 126), ('aouda', 125), ('train', 119), ('upon', 119), ('two', 107), ('sir', 103), ('master', 102), ('hundred', 97), ('twenty', 96), ('well', 96), ('replied', 93), ('hours', 90), ('day', 89)]


FR top 20: [('fogg', 671), ('passepartout', 445), ('phileas', 316), ('mr', 286), ('fix', 281), ('heures', 243), ('répondit', 214), ('dit', 183), ('deux', 151), ('monsieur', 145), ('même', 141), ('aouda', 136), ('quelques', 133), ('mrs', 131), ('là', 120), ('maître', 120), ('eût', 116), ('donc', 114), ('train', 114), ('mille', 108)]
Mêmes graphies dans les deux textes:


[('fogg', (584, 671)), ('passepartout', (392, 445)), ('mr', (390, 286)), ('phileas', (238, 316)), ('fix', (234, 281)), ('aouda', (125, 136)), ('train', (119, 114)), ('monsieur', (29, 145)), ('sir', (103, 55)), ('hong', (62, 66)), ('kong', (62, 66)), ('gentleman', (40, 87)), ('bombay', (61, 61)), ('moment', (46, 70)), ('minutes', (64, 51)), ('steamer', (88, 19)), ('francis', (53, 53)), ('point', (29, 74)), ('station', (48, 45)), ('club', (38, 54))]
Nom | EN | FR | EN / 10 000 | FR / 10 000


aouda 125 136 39.18 36.86
fix 234 281 73.35 76.16
fogg 584 671 183.07 181.87
passepartout 392 445 122.88 120.62
Interprétation : les fréquences relatives tiennent compte des tailles
différentes ; cette jointure ne traduit pas les mots.
